# Funciones de Ventana e Índices

### Window Functions — OVER()

In [108]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE employees (
    id INTEGER PRIMARY KEY, name TEXT, department TEXT, salary REAL)''')
cursor.executemany('INSERT INTO employees VALUES (?,?,?,?)', [
    (1, 'Alice', 'Engineering', 75000), (2, 'Bob', 'Marketing', 55000),
    (3, 'Carol', 'Engineering', 82000), (4, 'David', 'HR', 48000),
    (5, 'Eva', 'Marketing', 61000), (6, 'Frank', 'Engineering', 79000),
    (7, 'Grace', 'HR', 52000), (8, 'Henry', 'Marketing', 58000),
])
conn.commit()

# ROW_NUMBER, RANK, DENSE_RANK
cursor.execute("""
    SELECT
        id, name, department, salary,
        ROW_NUMBER() OVER (ORDER BY salary DESC)                        AS row_num,
        RANK()       OVER (ORDER BY salary DESC)                        AS rank_global,
        ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rank_in_dept
    FROM employees
    ORDER BY department desc
""")
print(f"{'Nombre':<10} {'Depto':<15} {'Salario':<15} {'Row#':>5} {'Rank':>5} {'DeptRk':>7}")
print("-" * 55)
for row in cursor.fetchall():
    # print(row)
    print(f"{row[0]:<4} | {row[1]:<10} | {row[2]:<12} | ${row[3]:<8} | {row[4]:<2} | {row[5]:<2} | {row[6]:<2} |")

Nombre     Depto           Salario          Row#  Rank  DeptRk
-------------------------------------------------------
5    | Eva        | Marketing    | $61000.0  | 4  | 4  | 1  |
8    | Henry      | Marketing    | $58000.0  | 5  | 5  | 2  |
2    | Bob        | Marketing    | $55000.0  | 6  | 6  | 3  |
7    | Grace      | HR           | $52000.0  | 7  | 7  | 1  |
4    | David      | HR           | $48000.0  | 8  | 8  | 2  |
3    | Carol      | Engineering  | $82000.0  | 1  | 1  | 1  |
6    | Frank      | Engineering  | $79000.0  | 2  | 2  | 2  |
1    | Alice      | Engineering  | $75000.0  | 3  | 3  | 3  |


### PARTITION BY — Ventanas por Grupo

In [109]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE sales (
    id INTEGER PRIMARY KEY, rep TEXT, region TEXT, amount REAL, sale_date TEXT)''')
cursor.executemany('INSERT INTO sales VALUES (?,?,?,?,?)', [
    (1, 'Alice', 'Norte', 5000, '2024-01-10'), (2, 'Bob', 'Sur', 3500, '2024-01-12'),
    (3, 'Carol', 'Norte', 7200, '2024-01-15'), (4,
                                                'David', 'Sur', 2800, '2024-01-18'),
    (5, 'Alice', 'Norte', 6100, '2024-02-05'), (6, 'Bob', 'Sur', 4200, '2024-02-08'),
    (7, 'Carol', 'Norte', 8900, '2024-02-12'), (8,
                                                'David', 'Sur', 3600, '2024-02-15'),
    (9, 'Eva',  'Este', 5500, '2024-02-20'), (10, 'Eva', 'Este', 4800, '2024-03-01'),
])
conn.commit()

# Porcentaje de cada venta dentro de su región
cursor.execute("""
    SELECT
        rep, region, amount,
        SUM(amount)   OVER (PARTITION BY region)                            AS region_total,
        ROUND(amount * 100.0 / SUM(amount) OVER (PARTITION BY region), 1)   AS pct_region,
        AVG(amount)   OVER (PARTITION BY region)                            AS region_avg,
        RANK()        OVER (PARTITION BY region ORDER BY amount DESC)       AS rank_in_region
    FROM sales
    ORDER BY region, amount DESC
""")
print(f"{'Rep':<8} {'Región':<6} {'Venta':>7} {'Total Reg':>10} {'%Reg':>6} {'Rank':>5}")
print("-" * 50)
for row in cursor.fetchall():
    print(
        f"| {row[0]:<8} | {row[1]:<6} | ${row[2]:>5,} | ${row[3]:>8,} | {row[4]:>5}% | {row[6]:>5}")

Rep      Región   Venta  Total Reg   %Reg  Rank
--------------------------------------------------
| Eva      | Este   | $5,500.0 | $10,300.0 |  53.4% |     1
| Eva      | Este   | $4,800.0 | $10,300.0 |  46.6% |     2
| Carol    | Norte  | $8,900.0 | $27,200.0 |  32.7% |     1
| Carol    | Norte  | $7,200.0 | $27,200.0 |  26.5% |     2
| Alice    | Norte  | $6,100.0 | $27,200.0 |  22.4% |     3
| Alice    | Norte  | $5,000.0 | $27,200.0 |  18.4% |     4
| Bob      | Sur    | $4,200.0 | $14,100.0 |  29.8% |     1
| David    | Sur    | $3,600.0 | $14,100.0 |  25.5% |     2
| Bob      | Sur    | $3,500.0 | $14,100.0 |  24.8% |     3
| David    | Sur    | $2,800.0 | $14,100.0 |  19.9% |     4


## Mine

In [110]:
import psycopg2

conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='movies', user='postgres', password='postgres'
)
cursor = conn.cursor()

cursor.execute(
    """
    Select 
        Concat(c.first_name, ' ', c.last_name),
        r.rental_date,
        p.amount as amount,
        SUM(amount) OVER(PARTITION BY r.rental_date) as total
    from customer AS c
    LEFT JOIN rental r ON c.customer_id = r.customer_id
    LEFT JOIN payment p on r.rental_id = p.rental_id
    ORDER BY r.rental_date
    """
)

for r in cursor.fetchall():
    print(r)

('CHARLOTTE HUNTER', datetime.datetime(2005, 5, 24, 19, 53, 30, tzinfo=datetime.timezone.utc), Decimal('2.99'), Decimal('2.99'))
('TOMMY COLLAZO', datetime.datetime(2005, 5, 24, 19, 54, 33, tzinfo=datetime.timezone.utc), Decimal('2.99'), Decimal('2.99'))
('MANUEL MURRELL', datetime.datetime(2005, 5, 24, 20, 3, 39, tzinfo=datetime.timezone.utc), Decimal('3.99'), Decimal('3.99'))
('ANDREW PURDY', datetime.datetime(2005, 5, 24, 20, 4, 41, tzinfo=datetime.timezone.utc), Decimal('4.99'), Decimal('4.99'))
('DELORES HANSEN', datetime.datetime(2005, 5, 24, 20, 5, 21, tzinfo=datetime.timezone.utc), Decimal('6.99'), Decimal('6.99'))
('NELSON CHRISTENSON', datetime.datetime(2005, 5, 24, 20, 8, 7, tzinfo=datetime.timezone.utc), Decimal('0.99'), Decimal('0.99'))
('CASSANDRA WALTERS', datetime.datetime(2005, 5, 24, 20, 11, 53, tzinfo=datetime.timezone.utc), Decimal('1.99'), Decimal('1.99'))
('MINNIE ROMERO', datetime.datetime(2005, 5, 24, 20, 31, 46, tzinfo=datetime.timezone.utc), Decimal('4.99'), D